# Exercício Prático – Visualização de Dados com Matplotlib

**Notebook completo com códigos e respostas no próprio arquivo.**

O exercício aborda gráficos básicos, personalização, `subplots`, `twinx`, anotações e heatmap com `imshow()`.


## Configuração do ambiente e geração dos dados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)
n_registros = 120

datas = pd.date_range(start='2024-01-01', periods=n_registros, freq='D')
categorias = ['Eletronicos', 'Livros', 'Roupas', 'Automotivo']
estados = ['SP', 'RJ', 'MG', 'RS', 'BA']

dados = {
    'data': datas,
    'categoria': np.random.choice(categorias, size=n_registros),
    'estado': np.random.choice(estados, size=n_registros),
    'valor': np.random.uniform(50, 4500, size=n_registros).round(2),
    'quantidade': np.random.randint(1, 15, size=n_registros),
}

df = pd.DataFrame(dados)
df['valor_venda'] = (df['valor'] * np.random.uniform(0.9, 1.1, size=n_registros)).round(2)
df.head()


# Parte 1 – Gráficos Essenciais e Personalização

## 5. Gráfico de linha – faturamento mensal

**Resposta:** O gráfico mostra o faturamento mensal total com marcadores circulares, linha tracejada, cor personalizada, grade e rótulos nos eixos.


In [ ]:
df = df.set_index('data')
faturamento_mensal = df['valor_venda'].resample('ME').sum()

plt.figure(figsize=(10,5))
plt.plot(faturamento_mensal.index, faturamento_mensal.values,
         marker='o', linestyle='--', color='royalblue')
plt.title('Faturamento Mensal Total')
plt.xlabel('Mês')
plt.ylabel('Faturamento (R$)')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Gráfico de barras – total de vendas por categoria

**Resposta:** As categorias são ordenadas do menor para o maior valor e os rótulos são rotacionados em 45 graus.


In [ ]:
vendas_categoria = df.groupby('categoria')['valor_venda'].sum().sort_values()

plt.figure(figsize=(9,5))
plt.bar(vendas_categoria.index, vendas_categoria.values)
plt.title('Total de Vendas por Categoria')
plt.xlabel('Categoria')
plt.ylabel('Total de Vendas (R$)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Histograma dos valores das vendas

**Resposta:** O histograma utiliza 10 bins e bordas pretas para destacar os intervalos.


In [ ]:
plt.figure(figsize=(9,5))
plt.hist(df['valor'], bins=10, edgecolor='black')
plt.title('Distribuição dos Valores das Vendas')
plt.xlabel('Valor da venda (R$)')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()


# Parte 2 – API Orientada a Objetos e Eixos Múltiplos

## 4–6. Subplots 2 x 1 com eixo X compartilhado

**Resposta:** O primeiro gráfico apresenta o faturamento mensal e o segundo apresenta a quantidade total de itens vendidos no mesmo período. Os dois utilizam o mesmo eixo X.


In [ ]:
faturamento_mensal = df['valor_venda'].resample('ME').sum()
quantidade_mensal = df['quantidade'].resample('ME').sum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11,8), sharex=True)

ax1.plot(faturamento_mensal.index, faturamento_mensal.values,
         marker='o', linestyle='--')
ax1.set_title('Faturamento Mensal')
ax1.set_ylabel('Faturamento (R$)')
ax1.grid(True)

ax2.bar(quantidade_mensal.index, quantidade_mensal.values)
ax2.set_title('Quantidade Total de Itens Vendidos por Mês')
ax2.set_xlabel('Mês')
ax2.set_ylabel('Quantidade')
ax2.grid(axis='y')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Dois eixos Y com `twinx()`

**Resposta:** O eixo Y esquerdo mostra o valor total vendido por mês em barras. O eixo Y direito mostra a quantidade média de itens por venda em uma linha com marcadores.


In [ ]:
valor_mensal = df['valor_venda'].resample('ME').sum()
quantidade_media_mensal = df['quantidade'].resample('ME').mean()

fig, ax = plt.subplots(figsize=(11,6))
ax.bar(valor_mensal.index, valor_mensal.values, width=20, alpha=0.7)
ax.set_xlabel('Mês')
ax.set_ylabel('Valor total vendido (R$)')
ax.set_title('Valor Total Vendido x Quantidade Média por Venda')
ax.tick_params(axis='x', rotation=45)

ax2 = ax.twinx()
ax2.plot(quantidade_media_mensal.index, quantidade_media_mensal.values,
         marker='o', linewidth=2)
ax2.set_ylabel('Quantidade média de itens por venda')

plt.tight_layout()
plt.show()


## 8. Anotação do mês de maior faturamento

**Resposta:** O mês de maior faturamento é encontrado com `idxmax()` e destacado com `ax.annotate()`, incluindo uma seta apontando para o pico.


In [ ]:
mes_maior = faturamento_mensal.idxmax()
valor_maior = faturamento_mensal.max()

fig, ax = plt.subplots(figsize=(11,6))
ax.plot(faturamento_mensal.index, faturamento_mensal.values,
        marker='o', linestyle='--')

ax.annotate(
    f'Maior faturamento\nR$ {valor_maior:,.2f}',
    xy=(mes_maior, valor_maior),
    xytext=(mes_maior, valor_maior * 0.85),
    arrowprops=dict(arrowstyle='->'),
    ha='center'
)

ax.set_title('Faturamento Mensal com Destaque para o Maior Faturamento')
ax.set_xlabel('Mês')
ax.set_ylabel('Faturamento (R$)')
ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Mês de maior faturamento:", mes_maior.strftime('%m/%Y'))
print(f"Valor do maior faturamento: R$ {valor_maior:,.2f}")


# Parte 3 – Matriz de Intensidade / Heatmap

## 7. Tabela dinâmica

**Resposta:** A `pivot_table` soma `valor_venda`, colocando as categorias nas linhas e os estados nas colunas.


In [ ]:
pivot_vendas = pd.pivot_table(
    df.reset_index(),
    values='valor_venda',
    index='categoria',
    columns='estado',
    aggfunc='sum'
)

display(pivot_vendas)


## 8. Heatmap com `imshow()`

**Resposta:** A matriz é representada visualmente por intensidade. Os estados aparecem no eixo X, as categorias no eixo Y e a barra de cores representa o valor das vendas.


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

imagem = ax.imshow(pivot_vendas.values, aspect='auto')

ax.set_xticks(np.arange(len(pivot_vendas.columns)))
ax.set_xticklabels(pivot_vendas.columns)

ax.set_yticks(np.arange(len(pivot_vendas.index)))
ax.set_yticklabels(pivot_vendas.index)

ax.set_xlabel('Estado')
ax.set_ylabel('Categoria')
ax.set_title('Heatmap de Vendas por Categoria e Estado')

plt.colorbar(imagem, ax=ax, label='Valor das vendas (R$)')
plt.tight_layout()
plt.show()


## 9. Salvando a figura

**Resposta:** O heatmap é salvo como `heatmap_vendas.png`, com resolução de 300 DPI e `bbox_inches='tight'`.


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

imagem = ax.imshow(pivot_vendas.values, aspect='auto')
ax.set_xticks(np.arange(len(pivot_vendas.columns)))
ax.set_xticklabels(pivot_vendas.columns)
ax.set_yticks(np.arange(len(pivot_vendas.index)))
ax.set_yticklabels(pivot_vendas.index)
ax.set_xlabel('Estado')
ax.set_ylabel('Categoria')
ax.set_title('Heatmap de Vendas por Categoria e Estado')

plt.colorbar(imagem, ax=ax, label='Valor das vendas (R$)')
plt.tight_layout()

plt.savefig('heatmap_vendas.png', dpi=300, bbox_inches='tight')
plt.show()

print('Arquivo salvo como: heatmap_vendas.png')
